In [8]:
import pandas as pd

df = pd.read_csv("../data/raw/game_logs.csv", dtype={"GAME_ID": str})
df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])
df["IS_HOME"] = df["MATCHUP"].str.contains("vs.", regex=False)

In [9]:
keep = ["GAME_ID", "GAME_DATE", "SEASON", "TEAM_ID", "TEAM_ABBREVIATION", "PTS", "WL"]

home = df[df["IS_HOME"]][keep].rename(columns={
    "TEAM_ID": "HOME_TEAM_ID",
    "TEAM_ABBREVIATION": "HOME_TEAM",
    "PTS": "HOME_PTS",
    "WL": "HOME_WL",
})

away = df[~df["IS_HOME"]][["GAME_ID", "TEAM_ID", "TEAM_ABBREVIATION", "PTS"]].rename(columns={
    "TEAM_ID": "AWAY_TEAM_ID",
    "TEAM_ABBREVIATION": "AWAY_TEAM",
    "PTS": "AWAY_PTS",
})

In [10]:
bad_ids = away.loc[away["GAME_ID"].duplicated(keep=False), "GAME_ID"].unique()

df[df["GAME_ID"].isin(bad_ids)][
    ["GAME_ID", "GAME_DATE", "TEAM_ABBREVIATION", "MATCHUP", "PTS", "WL"]
].sort_values(["GAME_DATE", "GAME_ID"])

,GAME_ID,GAME_DATE,TEAM_ABBREVIATION,MATCHUP,PTS,WL
21664,0022400147,2024-11-02,MIA,MIA @ WAS,118,W
21676,0022400147,2024-11-02,WAS,WAS @ MIA,98,L
22246,0022401229,2024-12-14,ATL,ATL @ MIL,102,L
22248,0022401229,2024-12-14,MIL,MIL @ ATL,110,W
22247,0022401230,2024-12-14,HOU,HOU @ OKC,96,L
22249,0022401230,2024-12-14,OKC,OKC @ HOU,111,W
22799,0022400621,2025-01-23,IND,IND @ SAS,110,L
22802,0022400621,2025-01-23,SAS,SAS @ IND,140,W
22829,0022400633,2025-01-25,IND,IND @ SAS,136,W
22833,0022400633,2025-01-25,SAS,SAS @ IND,98,L


In [11]:
away = away[~away["GAME_ID"].isin(bad_ids)]
games = home.merge(away, on="GAME_ID", validate="one_to_one")
games = games.sort_values("GAME_DATE").reset_index(drop=True)

In [12]:
games = home.merge(away, on="GAME_ID", validate="one_to_one")
games = games.sort_values("GAME_DATE").reset_index(drop=True)

In [13]:
games["HOME_WIN"] = (games["HOME_WL"] == "W").astype(int)
games = games.drop(columns=["HOME_WL"])

In [14]:
print("Rows:", len(games))
print("Unique games:", games["GAME_ID"].nunique())
print("Home win rate:", games["HOME_WIN"].mean())
print("Win matches points:", ((games["HOME_PTS"] > games["AWAY_PTS"]).astype(int) == games["HOME_WIN"]).all())
games.head()

Rows: 11974
Unique games: 11974
Home win rate: 0.5655587105395022
Win matches points: True


,GAME_ID,GAME_DATE,SEASON,HOME_TEAM_ID,HOME_TEAM,HOME_PTS,AWAY_TEAM_ID,AWAY_TEAM,AWAY_PTS,HOME_WIN
0,0021500001,2015-10-27,2015-16,1610612737,ATL,94,1610612765,DET,106,0
1,0021500002,2015-10-27,2015-16,1610612741,CHI,97,1610612739,CLE,95,1
2,0021500003,2015-10-27,2015-16,1610612744,GSW,111,1610612740,NOP,95,1
3,0021500007,2015-10-28,2015-16,1610612765,DET,92,1610612762,UTA,87,1
4,0021500011,2015-10-28,2015-16,1610612763,MEM,76,1610612739,CLE,106,0


In [15]:
games.to_csv("../data/processed/games.csv", index=False)